# 02 — Orchestration Patterns: Manager/Worker, Pipeline, Fan-out/Fan-in

This notebook reuses the exact `Message`/`MessageBus` primitives built and executed in
`14-multi-agent-systems/01-communication-protocols/001_message_bus_negotiation.ipynb`
(copied verbatim below, cited explicitly rather than silently duplicated) and builds three
orchestration patterns on top of them: **manager/worker (fan-out/fan-in)**,
**sequential pipeline**, and a look at where each breaks.

As in Topic 1: every agent below is a deterministic, scripted Python object. **No live LLM
API call is made anywhere in this notebook** — agents stand in for what an LLM-backed
worker or orchestrator would do, using plain rule-based logic instead, per this section's
binding constraint (see notes.md).

## Reused from Topic 1: `Message` and `MessageBus`

Copied unchanged from `01-communication-protocols/001_message_bus_negotiation.ipynb` — see that notebook for the original derivation and the direct/broadcast/blackboard experiments run against it.

In [1]:
from dataclasses import dataclass, field
from typing import Optional
import itertools

@dataclass
class Message:
    """A single agent-to-agent communication act. Identical to Topic 1's Message.

    sender / receiver: agent names (strings). receiver=None means the message
    is not addressed to one agent -- it's a broadcast or a blackboard post.
    performative: the FIPA-ACL-style speech act -- what kind of communicative
    intent this message carries.
    content: an arbitrary payload dict.
    """
    sender: str
    receiver: Optional[str]
    performative: str
    content: dict
    msg_id: int = field(default_factory=itertools.count().__next__)

    def __repr__(self):
        target = self.receiver if self.receiver else "ALL"
        return f"[{self.msg_id:02d}] {self.sender:>12} -> {target:<12} {self.performative:<10} {self.content}"


In [2]:
class MessageBus:
    """Plain-Python, in-process message router. No network, no threads.
    Identical to Topic 1's MessageBus -- see 01-communication-protocols/notes.md
    for the full direct/broadcast/blackboard discussion.
    """
    def __init__(self):
        self.agents = {}
        self.blackboard = []
        self.log = []  # every message ever sent through this bus, in order

    def register(self, agent):
        self.agents[agent.name] = agent
        agent.bus = self

    def send(self, msg: Message):
        self.log.append(msg)
        if msg.receiver is None:
            raise ValueError("send() requires a receiver; use broadcast() for None")
        target = self.agents.get(msg.receiver)
        if target is None:
            raise KeyError(f"unknown receiver {msg.receiver!r}")
        target.receive(msg)

    def broadcast(self, msg: Message, exclude_sender=True):
        self.log.append(msg)
        for name, agent in self.agents.items():
            if exclude_sender and name == msg.sender:
                continue
            agent.receive(msg)


class Agent:
    def __init__(self, name):
        self.name = name
        self.bus = None
        self.inbox = []

    def receive(self, msg: Message):
        self.inbox.append(msg)


## The task: distributed sum with a checkable correct answer

Every pattern below solves the *same* toy task, so the patterns can be compared fairly:
split a list of `N` numbers into chunks, have workers sum their chunk, combine the partial
sums into a total, and check the total against Python's own built-in `sum()`. `sum()` is
not part of any agent's code — it is the independent ground truth used only to verify
correctness after the fact.

In [3]:
def split_into_chunks(numbers, n_chunks):
    """Contiguous split into n_chunks nearly-equal pieces (last chunk absorbs the remainder)."""
    n = len(numbers)
    base = n // n_chunks
    chunks = []
    start = 0
    for i in range(n_chunks):
        extra = 1 if i < (n % n_chunks) else 0
        end = start + base + extra
        chunks.append(numbers[start:end])
        start = end
    assert sum(len(c) for c in chunks) == n
    return chunks

NUMBERS = list(range(1, 998))  # 997 numbers, deliberately not a round count
print("N =", len(NUMBERS), " true sum (ground truth) =", sum(NUMBERS))


N = 997  true sum (ground truth) = 497503


## Pattern 1: Manager/worker (fan-out / fan-in orchestration)

**Task decomposition.** The `Orchestrator` splits `NUMBERS` into one chunk per `Worker` and
sends each worker a direct `assign` message carrying only that worker's chunk.

**Delegation.** Each `assign` message is logically independent of every other — no worker
needs to see any other worker's chunk or result before it can start.

**Worker execution.** Each `Worker.receive()` computes `sum(chunk)` (its "work") and replies
directly to the orchestrator with an `inform` message carrying its `partial_sum`.

**Result aggregation.** The orchestrator collects every `partial_sum` and sums them locally
— `aggregate()` is the only place the final answer is assembled.

Every message also carries a `"round"` field — an explicit **logical clock**, not a
wall-clock timestamp. `round=0` is the orchestrator's dispatch step; every worker computes
its own round as `1 + incoming.round`. Because every worker receives its `assign` message at
the *same* round (0) and works from it independently, every worker's reply also lands at the
*same* round (1) — this is how the notebook will measure "how many sequential dependency
steps does this pattern need," honestly, without claiming a wall-clock speedup this
single-process, synchronous bus cannot actually produce (see "Honesty about measurement"
below).

In [4]:
class Worker(Agent):
    """Executes one assigned chunk and reports a partial result back to whoever assigned it."""
    def receive(self, msg):
        super().receive(msg)
        if msg.performative == "assign":
            chunk = msg.content["chunk"]
            partial_sum = sum(chunk)  # the worker's "computation" -- deterministic, no LLM
            reply = Message(sender=self.name, receiver=msg.sender, performative="inform",
                             content={"partial_sum": partial_sum, "n_items": len(chunk),
                                      "round": msg.content["round"] + 1})
            self.bus.send(reply)


class Orchestrator(Agent):
    """Decomposes a task across workers, delegates it, and aggregates their results."""
    def __init__(self, name):
        super().__init__(name)
        self.partial_sums = []
        self.n_items_seen = 0
        self.max_round_seen = 0
        self.final_result = None

    def receive(self, msg):
        super().receive(msg)
        if msg.performative == "inform":
            self.partial_sums.append(msg.content["partial_sum"])
            self.n_items_seen += msg.content["n_items"]
            self.max_round_seen = max(self.max_round_seen, msg.content["round"])

    def decompose_and_delegate(self, numbers, workers):
        chunks = split_into_chunks(numbers, len(workers))
        for w, chunk in zip(workers, chunks):
            msg = Message(sender=self.name, receiver=w.name, performative="assign",
                           content={"chunk": chunk, "round": 0})
            self.bus.send(msg)

    def aggregate(self, validate=True):
        if validate:
            for p in self.partial_sums:
                if not isinstance(p, (int, float)):
                    raise TypeError(f"worker returned a non-numeric partial_sum: {p!r}")
        self.final_result = sum(self.partial_sums)
        self.aggregate_round = self.max_round_seen + 1
        return self.final_result


In [5]:
def run_fan_out_fan_in(numbers, n_workers):
    bus = MessageBus()
    orch = Orchestrator("orchestrator")
    workers = [Worker(f"worker_{i}") for i in range(n_workers)]
    for a in [orch] + workers:
        bus.register(a)
    orch.decompose_and_delegate(numbers, workers)
    total = orch.aggregate()
    return bus, orch, total

bus, orch, total = run_fan_out_fan_in(NUMBERS, n_workers=4)

print("--- transcript (fan-out/fan-in, 4 workers) ---")
for m in bus.log:
    print(m)
print()
print("orchestrator aggregate:", total)
print("python sum() ground truth:", sum(NUMBERS))
print("aggregate happened at logical round:", orch.aggregate_round)
assert total == sum(NUMBERS)
print("CORRECT: orchestrator total matches sum() exactly")


--- transcript (fan-out/fan-in, 4 workers) ---
[00] orchestrator -> worker_0     assign     {'chunk': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 

## Pattern 2: Sequential pipeline

Contrast pattern: the *same* task, but each worker's output feeds the next instead of every
worker executing independently in parallel. A `PipelineWorker` is constructed with its own
fixed chunk and the name of the *next* agent in the chain. It does not reply to the
orchestrator — it adds its chunk's sum to a running `accumulator` carried inside the message
and forwards that message to the next worker. Only the last worker in the chain hands the
final accumulator back to the orchestrator.

This is a genuine dependency chain: `worker_2` cannot even start until `worker_1`'s message
arrives, because the message `worker_2` needs (the running accumulator) does not exist until
`worker_1` produces it. Compare this to Pattern 1, where every `Worker` could in principle
start the instant it receives its `assign` message, with no dependency on any other
worker.

In [6]:
class PipelineWorker(Agent):
    """Holds one fixed chunk; on receiving the running accumulator, adds its chunk's sum
    and forwards to the next agent in the chain (the next worker, or the orchestrator if
    this is the last stage)."""
    def __init__(self, name, chunk, next_name):
        super().__init__(name)
        self.chunk = chunk
        self.next_name = next_name

    def receive(self, msg):
        super().receive(msg)
        if msg.performative == "accumulate":
            new_acc = msg.content["accumulator"] + sum(self.chunk)
            reply = Message(sender=self.name, receiver=self.next_name, performative="accumulate",
                             content={"accumulator": new_acc, "round": msg.content["round"] + 1})
            self.bus.send(reply)


class PipelineOrchestrator(Agent):
    def __init__(self, name):
        super().__init__(name)
        self.final_result = None
        self.final_round = None

    def receive(self, msg):
        super().receive(msg)
        if msg.performative == "accumulate":
            self.final_result = msg.content["accumulator"]
            self.final_round = msg.content["round"]

    def run(self, numbers, n_workers):
        chunks = split_into_chunks(numbers, n_workers)
        names = [f"pworker_{i}" for i in range(n_workers)]
        workers = []
        for i in range(n_workers):
            next_name = names[i + 1] if i + 1 < n_workers else self.name
            workers.append(PipelineWorker(names[i], chunks[i], next_name))
        for a in workers:
            self.bus.register(a)
        kickoff = Message(sender=self.name, receiver=names[0], performative="accumulate",
                           content={"accumulator": 0, "round": 0})
        self.bus.send(kickoff)
        return self.final_result


In [7]:
def run_pipeline(numbers, n_workers):
    bus = MessageBus()
    porch = PipelineOrchestrator("orchestrator")
    bus.register(porch)
    total = porch.run(numbers, n_workers)
    return bus, porch, total

pbus, porch, ptotal = run_pipeline(NUMBERS, n_workers=4)

print("--- transcript (sequential pipeline, 4 workers) ---")
for m in pbus.log:
    print(m)
print()
print("pipeline final result:", ptotal)
print("python sum() ground truth:", sum(NUMBERS))
print("final result arrived at logical round:", porch.final_round)
assert ptotal == sum(NUMBERS)
print("CORRECT: pipeline total matches sum() exactly")


--- transcript (sequential pipeline, 4 workers) ---
[08] orchestrator -> pworker_0    accumulate {'accumulator': 0, 'round': 0}
[09]    pworker_0 -> pworker_1    accumulate {'accumulator': 31375, 'round': 1}
[10]    pworker_1 -> pworker_2    accumulate {'accumulator': 124750, 'round': 2}
[11]    pworker_2 -> pworker_3    accumulate {'accumulator': 280126, 'round': 3}
[12]    pworker_3 -> orchestrator accumulate {'accumulator': 497503, 'round': 4}

pipeline final result: 497503
python sum() ground truth: 497503
final result arrived at logical round: 4
CORRECT: pipeline total matches sum() exactly


## Honesty about measurement

This bus is synchronous, single-threaded, and single-process (unchanged from Topic 1) —
every `bus.send()` call blocks until the receiving agent's `receive()` returns. That means
a naive `time.perf_counter()` comparison between Pattern 1 and Pattern 2 would measure
**Python function-call overhead and chunk size**, not the thing that actually differs
between fan-out/fan-in and sequential pipelines in a real deployment (independent workers
that *could* run concurrently vs. a chain that structurally cannot). Wrapping the workers in
`threading` would not fix this either — CPython's GIL serializes pure-Python `sum()` work
regardless of thread count, so a thread-based "parallel" run would not show a real speedup
for this CPU-bound task, and a `multiprocessing`/`concurrent.futures.ProcessPoolExecutor`
version would mostly measure process-spawn overhead for a task this small (that overhead is
itself instructive — see "Failure modes" → over-decomposition below — but it would not be
an honest measurement of the *orchestration pattern's* efficiency).

**What is measured instead: logical rounds** — the number of *sequential dependency steps*
required to reach the final answer, computed from the causal `round` field each message
already carries (not a stopwatch). A "round" only advances when a message's payload
*depends on* the content of a previous message. This is exactly the metric that
distinguishes the two patterns structurally: it answers "how many steps would this pattern
need if every independent worker actually ran at the same time," which is the real claim
fan-out/fan-in orchestration makes, without asserting a wall-clock number this
single-process notebook cannot honestly produce.

In [8]:
def logical_rounds_fan_out(n_workers):
    _, orch, total = run_fan_out_fan_in(NUMBERS, n_workers)
    assert total == sum(NUMBERS)
    return orch.aggregate_round

def logical_rounds_pipeline(n_workers):
    _, porch, total = run_pipeline(NUMBERS, n_workers)
    assert total == sum(NUMBERS)
    return porch.final_round

worker_counts = [2, 4, 8, 16, 32]
print("{:>10} {:>16} {:>18}".format("n_workers", "fan_out_rounds", "pipeline_rounds"))
rows = []
for n in worker_counts:
    fo = logical_rounds_fan_out(n)
    pl = logical_rounds_pipeline(n)
    rows.append((n, fo, pl))
    print(f"{n:>10} {fo:>16} {pl:>18}")

# hypothesis: fan-out/fan-in stays constant (O(1)) as n grows; pipeline grows linearly (O(n))
for n, fo, pl in rows:
    assert fo == 2, f"fan-out rounds should be constant at 2, got {fo} for n={n}"
    assert pl == n, f"pipeline rounds should equal n, got {pl} for n={n}"
print()
print("hypothesis confirmed: fan-out/fan-in rounds = 2 (constant); pipeline rounds = n (linear)")


 n_workers   fan_out_rounds    pipeline_rounds
         2                2                  2
         4                2                  4
         8                2                  8
        16                2                 16
        32                2                 32

hypothesis confirmed: fan-out/fan-in rounds = 2 (constant); pipeline rounds = n (linear)


**Interpretation.** Fan-out/fan-in needs exactly 2 logical rounds to reach the final
answer no matter how many workers are involved (round 1: every worker computes from the same
dispatch; round 2: the orchestrator aggregates) — this is the structural reason manager/worker
orchestration is described as parallelizable: nothing in the causal message graph forces
worker $i$ to wait on worker $j$. The sequential pipeline needs exactly $n$ logical rounds,
growing linearly with the number of workers, because each stage's input is literally the
previous stage's output. This is a real, measured, honestly-computed difference — it is just
a difference in *dependency depth*, not wall-clock seconds, and that distinction is stated
explicitly rather than glossed over.

A secondary, and much weaker, metric is raw message count — shown next for completeness, but
it does *not* distinguish the two patterns the way rounds do.

In [9]:
def message_count_fan_out(n_workers):
    bus, orch, total = run_fan_out_fan_in(NUMBERS, n_workers)
    return len(bus.log)

def message_count_pipeline(n_workers):
    bus, porch, total = run_pipeline(NUMBERS, n_workers)
    return len(bus.log)

print("{:>10} {:>14} {:>16}".format("n_workers", "fan_out_msgs", "pipeline_msgs"))
for n in worker_counts:
    print(f"{n:>10} {message_count_fan_out(n):>14} {message_count_pipeline(n):>16}")
print()
print("fan-out/fan-in sends 2n messages (n assigns + n informs); pipeline sends n+1 (one")
print("accumulate hop per stage, including the kickoff). Both are LINEAR in n -- message")
print("count alone does not show the O(1)-vs-O(n) structural difference that logical rounds")
print("did; it only shows fan-out/fan-in has a larger constant factor for the same worker")
print("count, for the opposite reason (n independent replies vs n chained hops). This is")
print("the same lesson Topic 1's experiment taught about broadcast vs direct: counting")
print("messages and counting real structural cost are not the same measurement.")


 n_workers   fan_out_msgs    pipeline_msgs
         2              4                3
         4              8                5
         8             16                9
        16             32               17
        32             64               33

fan-out/fan-in sends 2n messages (n assigns + n informs); pipeline sends n+1 (one
accumulate hop per stage, including the kickoff). Both are LINEAR in n -- message
count alone does not show the O(1)-vs-O(n) structural difference that logical rounds
did; it only shows fan-out/fan-in has a larger constant factor for the same worker
count, for the opposite reason (n independent replies vs n chained hops). This is
the same lesson Topic 1's experiment taught about broadcast vs direct: counting
messages and counting real structural cost are not the same measurement.


## Failure modes

### 1. A worker returns a malformed result — no validation catches it until it crashes

`Orchestrator.aggregate(validate=True)` type-checks every `partial_sum` before summing. Turn
that check off and see what a single malformed worker reply does to the whole aggregation.

In [10]:
class MalformedWorker(Worker):
    """Simulates a broken worker -- e.g. a bug that serializes the result as a string,
    or an LLM-backed worker (in a real system) that returned prose instead of a number."""
    def receive(self, msg):
        if msg.performative == "assign":
            reply = Message(sender=self.name, receiver=msg.sender, performative="inform",
                             content={"partial_sum": "sorry, I could not compute that",
                                      "n_items": len(msg.content["chunk"]),
                                      "round": msg.content["round"] + 1})
            self.bus.send(reply)

def run_with_one_malformed_worker(numbers, n_workers, validate):
    bus = MessageBus()
    orch = Orchestrator("orchestrator")
    workers = [Worker(f"worker_{i}") for i in range(n_workers - 1)]
    workers.append(MalformedWorker(f"worker_{n_workers - 1}"))
    for a in [orch] + workers:
        bus.register(a)
    orch.decompose_and_delegate(numbers, workers)
    return orch.aggregate(validate=validate)

print("-- without validation (validate=False) --")
try:
    bad_total = run_with_one_malformed_worker(NUMBERS, 4, validate=False)
    print("aggregate() returned:", bad_total)
except TypeError as e:
    print("crashed with TypeError:", e)

print()
print("-- with validation (validate=True) --")
try:
    run_with_one_malformed_worker(NUMBERS, 4, validate=True)
except TypeError as e:
    print("caught cleanly at aggregation time:", e)


-- without validation (validate=False) --
crashed with TypeError: unsupported operand type(s) for +: 'int' and 'str'

-- with validation (validate=True) --
caught cleanly at aggregation time: worker returned a non-numeric partial_sum: 'sorry, I could not compute that'


Without validation, `sum()` on a list containing one string simply raises `TypeError`
deep inside Python's own summation — a crash, but a *confusing* one: nothing in the traceback
points at which worker produced the bad value or why. With the type check in place, the
orchestrator fails at the same point but with a message that names the actual malformed
payload, right where the bad data entered the system. This is already an improvement, but it
is a narrow one — it only catches results that are the *wrong type*. The next example shows a
malformed result that is the *right type* and passes that same check while still being
wrong.

### 2. A worker returns a wrong-but-well-typed result — silently poisons the total

A type check cannot catch a worker that is confidently, plausibly, and incorrectly wrong.

In [11]:
class OffByBugWorker(Worker):
    """Simulates a subtler bug: a real-looking numeric result that is simply wrong
    (e.g. an off-by-one in a hand-rolled loop, or an LLM-backed worker that miscounted).
    Always under-reports by a fixed amount -- a plausible bug shape, not an obvious crash."""
    def receive(self, msg):
        if msg.performative == "assign":
            chunk = msg.content["chunk"]
            wrong_sum = sum(chunk) - 100  # plausible-looking, deterministic, wrong
            reply = Message(sender=self.name, receiver=msg.sender, performative="inform",
                             content={"partial_sum": wrong_sum, "n_items": len(chunk),
                                       "round": msg.content["round"] + 1})
            self.bus.send(reply)

bus = MessageBus()
orch = Orchestrator("orchestrator")
workers = [Worker("worker_0"), Worker("worker_1"), Worker("worker_2"), OffByBugWorker("worker_3")]
for a in [orch] + workers:
    bus.register(a)
orch.decompose_and_delegate(NUMBERS, workers)
poisoned_total = orch.aggregate(validate=True)  # type check passes -- wrong_sum IS an int

print("orchestrator total (one buggy worker):", poisoned_total)
print("python sum() ground truth:            ", sum(NUMBERS))
print("silently wrong by:", sum(NUMBERS) - poisoned_total)
print("type validation raised no error:", isinstance(poisoned_total, int))
assert poisoned_total != sum(NUMBERS)
print()
print("this is the dangerous case: aggregate() returned a real int, validate=True passed,")
print("and the final answer is simply wrong -- exactly as many wrong-answer worker replies")
print("(hallucinated numbers, miscounted results) would look in a real LLM-backed deployment.")
print("catching THIS requires a redundant check (e.g. re-deriving n_items_seen == len(numbers),")
print("or spot-checking against an independent computation) -- a type check alone is not enough.")


orchestrator total (one buggy worker): 497403
python sum() ground truth:             497503
silently wrong by: 100
type validation raised no error: True

this is the dangerous case: aggregate() returned a real int, validate=True passed,
and the final answer is simply wrong -- exactly as many wrong-answer worker replies
(hallucinated numbers, miscounted results) would look in a real LLM-backed deployment.
catching THIS requires a redundant check (e.g. re-deriving n_items_seen == len(numbers),
or spot-checking against an independent computation) -- a type check alone is not enough.


In [12]:
# The n_items sanity check the Orchestrator already tracks *does* catch a different class
# of bug -- a worker that drops or duplicates items -- even though it could not catch the
# off-by-a-constant bug above (that bug preserved n_items exactly).
class DroppingWorker(Worker):
    """Silently drops the last element of its chunk before summing -- n_items now disagrees."""
    def receive(self, msg):
        if msg.performative == "assign":
            chunk = msg.content["chunk"][:-1]
            reply = Message(sender=self.name, receiver=msg.sender, performative="inform",
                             content={"partial_sum": sum(chunk), "n_items": len(msg.content["chunk"]),
                                       "round": msg.content["round"] + 1})
            self.bus.send(reply)
            # NOTE: n_items reported is the ASSIGNED chunk length, not the ACTUAL items summed --
            # this models a worker that mis-reports its own metadata, which n_items_seen alone
            # cannot catch either. A dropped item that is honestly reported...

class HonestDroppingWorker(Worker):
    """Same bug, but honestly reports how many items it actually summed -- this IS catchable."""
    def receive(self, msg):
        if msg.performative == "assign":
            chunk = msg.content["chunk"][:-1]
            reply = Message(sender=self.name, receiver=msg.sender, performative="inform",
                             content={"partial_sum": sum(chunk), "n_items": len(chunk),
                                       "round": msg.content["round"] + 1})
            self.bus.send(reply)

bus = MessageBus()
orch = Orchestrator("orchestrator")
workers = [Worker("worker_0"), Worker("worker_1"), Worker("worker_2"), HonestDroppingWorker("worker_3")]
for a in [orch] + workers:
    bus.register(a)
orch.decompose_and_delegate(NUMBERS, workers)
total = orch.aggregate(validate=True)
items_check_passes = (orch.n_items_seen == len(NUMBERS))

print("total:", total, " ground truth:", sum(NUMBERS), " equal:", total == sum(NUMBERS))
print("n_items_seen:", orch.n_items_seen, " len(NUMBERS):", len(NUMBERS),
      " n_items sanity check passes:", items_check_passes)
assert total != sum(NUMBERS)
assert not items_check_passes
print()
print("here the redundant n_items check DOES catch the wrong total (n_items_seen != len(NUMBERS)),")
print("because this bug happens to corrupt metadata the orchestrator can cross-check. The earlier")
print("OffByBugWorker bug did not corrupt n_items, so the same check passed right over it. No single")
print("sanity check catches every wrong-worker failure mode -- what gets caught depends entirely on")
print("which invariant the specific bug happens to violate.")


total: 496506  ground truth: 497503  equal: False
n_items_seen: 996  len(NUMBERS): 997  n_items sanity check passes: False

here the redundant n_items check DOES catch the wrong total (n_items_seen != len(NUMBERS)),
because this bug happens to corrupt metadata the orchestrator can cross-check. The earlier
OffByBugWorker bug did not corrupt n_items, so the same check passed right over it. No single
sanity check catches every wrong-worker failure mode -- what gets caught depends entirely on
which invariant the specific bug happens to violate.


### 3. Orchestrator as single point of failure / bottleneck

Every pattern above routes **all** delegation and **all** aggregation through one
`Orchestrator` object. If that object's logic is wrong (as `aggregate()`'s validation gap
just showed) or it becomes unavailable, the entire task stalls or silently corrupts — no
worker in either pattern has any way to detect or route around a broken orchestrator, because
no worker ever talks to another worker directly (fan-out/fan-in) or to anyone but its
immediate neighbor (pipeline). This mirrors Topic 1's blackboard-vs-direct tradeoff in
reverse: pushing all coordination through one node makes reasoning about the system easy
(there is exactly one place aggregation logic lives) at the cost of exactly one place where
everything can break.

### 4. Over-decomposition: coordination overhead exceeds the parallelism benefit

The "logical rounds" experiment above showed fan-out/fan-in staying at a constant 2 rounds
regardless of `n_workers` — but that used the *causal dependency depth*, which does not
capture per-worker overhead (constructing a chunk, building and dispatching a `Message`,
registering a worker). The message-count experiment showed the real, measured cost that
*does* scale with `n_workers`: **2n messages** for fan-out/fan-in (`n` assigns + `n` informs),
exactly as counted above (not estimated). If the total work (997 numbers) is split across more
and more workers, each worker's actual payload keeps shrinking while the fixed 2-message-per-
worker coordination cost keeps growing — at `n_workers=997` (one number per worker) there are
1,994 coordination messages moving a grand total of one addition each. The real parallelism
benefit (constant rounds) is unchanged, but the coordination cost this notebook can actually
count has grown to dominate the tiny amount of real work being delegated. This is the concrete,
measured version of "orchestration overhead exceeds the benefit of decomposition" — the numbers
above (2n messages, growing linearly and unboundedly, against a fixed task size of 997 items)
are the evidence, not an assertion.

## Real-world usage

- **LangGraph** (LangChain) models multi-agent workflows as an explicit graph of nodes and
  edges, where a node can be an LLM call, a tool call, or a sub-graph — the manager/worker
  pattern in this notebook (`Orchestrator.decompose_and_delegate` → parallel `Worker` nodes →
  `aggregate`) maps directly onto a LangGraph "map" over parallel branches that join back into
  one aggregation node, and the sequential pipeline maps onto a simple linear chain of nodes.
- **CrewAI's hierarchical process** puts a dedicated "manager" agent in charge of breaking a
  goal into tasks, assigning each task to the crew member best suited for it, and reviewing
  results before finishing — structurally the same decompose → delegate → aggregate loop as
  this notebook's `Orchestrator`, except CrewAI's manager and workers are real LLM calls
  making those decisions dynamically at each step, rather than the fixed, scripted
  `split_into_chunks` and `sum()` used here.
- **AutoGen's group chat manager** coordinates multiple conversable agents in a shared
  conversation, deciding at each turn which agent should speak next — closer to Topic 1's
  broadcast topology plus a scheduling policy than to this notebook's direct manager/worker
  delegation, but it plays the same "central coordinator" role this notebook's
  `Orchestrator` plays: one component with the authority to decide who acts next and how
  results get combined.

None of these three frameworks are called, imported, or executed anywhere in this
section — named here only to connect this notebook's toy `Orchestrator`/`Worker`/
`PipelineWorker` abstractions to real, production orchestration designs by name, the same
substitution Topic 1's notes.md documents and this section's binding constraints require.

## Summary of what was actually measured in this notebook

- Both patterns produce a total that matches Python's `sum()` exactly, for `n_workers` in
  `{2, 4, 8, 16, 32}` — asserted, not eyeballed.
- Fan-out/fan-in needs a constant 2 logical rounds regardless of worker count; the sequential
  pipeline needs exactly `n_workers` logical rounds — asserted for every tested `n`.
- Both patterns send exactly `2 * n_workers` bus messages for this task — message count alone
  does *not* distinguish them; only the dependency-depth (rounds) metric does.
- A malformed (wrong-type) worker result is caught by a type check at aggregation time; a
  wrong-but-well-typed worker result is not caught by that same check, and silently produces
  an incorrect total — demonstrated concretely, not just described.
- A redundant `n_items_seen` cross-check catches one class of worker bug (dropped/duplicated
  items honestly reported) but not another (a plausible-looking wrong value that preserves
  item count) — no single sanity check is sufficient on its own.